# 00 — Build outage events in Colab (Lights Out)

This notebook does the one heavy step on Google's machines, so nothing large touches your laptop:

1. Downloads the small datasets (NOAA storms, CDC SVI, zone→county file, county shapes, EAGLE-I customer counts) into Drive, about 110 MB.
2. For each year 2014–2022, downloads one EAGLE-I file (~1 GB) to Colab's **temporary** disk, converts it into outage events, saves the small result to Drive, and deletes the raw file.
3. Matches storms to outages, joins SVI, and writes `data/processed/model_table.parquet`.

**Setup (once):** upload `LightsOut_code.zip` to the top level of *My Drive*. Then **Runtime → Run all**. Expect about 20–40 minutes; the free tier is enough.

Outputs in `My Drive/LightsOut/data/`:
- `interim/outage_events_YYYY.parquet`: one row per county outage event
- `interim/daily_YYYY.parquet`: per county-day max customers out
- `interim/case_*.parquet`: hourly series for Texas Feb 2021 and Hurricane Ida
- `processed/model_table.parquet`: the (storm episode, county) modeling table

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, zipfile, pathlib
PROJECT = pathlib.Path('/content/drive/MyDrive/LightsOut')
ZIP = pathlib.Path('/content/drive/MyDrive/LightsOut_code.zip')
# always unpack the latest uploaded code (src/, notebooks/); data/ is never touched
assert ZIP.exists(), 'Upload LightsOut_code.zip to the top level of My Drive first.'
zipfile.ZipFile(ZIP).extractall(PROJECT)
print(sorted(p.name for p in PROJECT.iterdir()))

In [ ]:
!pip -q install polars pyarrow geopandas

# Big EAGLE-I CSVs go to Colab's local disk (deleted after each year), not Drive
os.environ['LIGHTSOUT_EAGLEI_RAW'] = '/content/eaglei_raw'
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

from lightsout import config as C, download, eaglei, noaa, match, features
print('raw (small):', C.RAW)
print('raw EAGLE-I (temporary):', C.EAGLEI_RAW)
print('outputs:', C.INTERIM, C.PROCESSED)

## 1. Small datasets → Drive

In [ ]:
download.main()  # NOAA, zone-county, SVI, county shapes, EAGLE-I MCC/DQI/coverage

## 2. EAGLE-I, one year at a time

Each year needs the **next** year's file too (the first 3 days are used to finish restorations that cross New Year), so at most two raw files are on disk at once. Years already processed with the current pipeline version are skipped (older outputs are rebuilt automatically), so if Colab disconnects, just re-run this cell.

In [ ]:
import time
YEARS = C.YEARS  # 2014-2022; use e.g. [2020, 2021, 2022] for a quick midterm run

def raw(y):
    return C.EAGLEI_RAW / f'eaglei_outages_{y}.csv'

for y in YEARS:
    if eaglei.is_current(y):
        print(f'{y}: already processed with the current pipeline, skipping'); continue
    t = time.time()
    download.eaglei_year(y)
    if y + 1 <= max(C.YEARS):
        download.eaglei_year(y + 1)
    eaglei.run(y)
    raw(y).unlink(missing_ok=True)           # free Colab disk
    print(f'   ({time.time() - t:.0f}s)')

for p in C.EAGLEI_RAW.glob('*.csv'):
    p.unlink()

## 3. Storms → (episode, county) units → matching → modeling table

In [ ]:
noaa.main()
match.main()
features.main()

## 4. Sanity checks

In [ ]:
import pandas as pd
ev = pd.concat(pd.read_parquet(p) for p in sorted(C.INTERIM.glob('outage_events_*.parquet')))
print(ev.groupby('year').agg(events=('fips','size'), counties=('fips','nunique'),
                             median_restore_h=('restore_hours','median'),
                             censored=('censored','mean')).round(3))

# Harris County, TX (48201) during the Feb 2021 winter storm: expect > 1M customers out
tx = ev[(ev.fips == 48201) & (ev.start.between('2021-02-10', '2021-02-25'))]
print(tx[['start','peak','peak_pct','restore_hours','censored']])

# Orleans (22071) and Jefferson (22051) Parish, Hurricane Ida
ida = ev[ev.fips.isin([22071, 22051]) & ev.start.between('2021-08-28', '2021-09-15')]
print(ida[['fips','start','peak','peak_pct','restore_hours','censored']])

mt = pd.read_parquet(C.PROCESSED / 'model_table.parquet')
assert not mt.duplicated(['episode_id','fips']).any()
assert mt.peak_pct.between(0, 1).all()
assert (mt.loc[mt.had_outage, 'restore_hours'] >= 0).all()
print(mt.groupby('split').size())

## Done

Everything downstream (`01_eda` … `04_equity`) only needs the small files in `My Drive/LightsOut/data/`. Run those notebooks here in Colab, or download the `data/interim` and `data/processed` folders (tens of MB) to the same paths on your laptop.